# Python for Data Engineering

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/python.ipynb)

Python is the default glue language of data engineering: orchestrators are written in it, engines expose a first-class API, and most pipeline code is Python even when the compute runs in Spark or a warehouse. The interview bar is not algorithmic trivia; it is whether you can write pipeline code that is memory-safe, idempotent, observable, testable, and reproducibly packaged.

Three themes run through this page: **scale** (processing more than memory allows), **correctness** (idempotent writes, retries, dead-letter handling), and **operability** (structured logs, pinned environments, CI tests).

## Why is Python widely used in data engineering?

Python is popular because it is productive, readable, and has a strong ecosystem for data processing, orchestration, and integration. It is commonly used for glue code, transformations, APIs, and automation, even when the heavy compute runs on distributed engines.

What that looks like in a real stack: **orchestration** (Airflow, Dagster), **transformation** (PySpark, Polars, pandas, SQLAlchemy), **ingestion** (`httpx`, `boto3`, Kafka clients), and **testing/CI** (pytest, Great Expectations). The trade-off to name: Python is slow per CPU-cycle and heavy per process, so the winning pattern is "Python for control flow and small data, native or distributed engines for heavy lifting."

In [ ]:
try:
    # Python usually plays the control-plane role: orchestrate, do not crunch
    from airflow.sdk import dag, task


    @dag(dag_id="orders_elt", schedule="0 3 * * *", start_date=None, tags=["elt"])
    def orders_elt():
        @task
        def extract() -> str:
            return extract_to_s3("orders", "2026-10-04")

        @task
        def transform(path: str) -> int:
            return run_spark_job("jobs/transform_orders.py", path)

        transform(extract())


    orders_elt()
except ModuleNotFoundError as exc:
    print(f"SKIPPED - {getattr(exc, 'name', None) or 'module'} is not installed; install it and re-run this cell.")

**Mechanism.** Python is the control plane, not the compute engine. The interpreter decides *what* to run and calls into C, Java, or Rust code for the heavy parts, so a short Python file can drive a cluster-wide job.

**Worked example.** A daily job is 200 lines of Python: list the partitions, submit a Spark job per partition, poll a control table, write a status row. The row math runs inside Spark; Python only moves names and paths.

**Common mistake.** Rewriting the transform as a Python loop over a million rows. It runs, but it is far slower and pulls the data into memory. Push work down to SQL or Spark and keep Python for control flow.

## How do iterators and generators help with large data processing?

Iterators and generators allow streaming data instead of loading everything into memory. They enable processing large datasets row-by-row or batch-by-batch. This reduces memory usage and often improves stability of ETL jobs.

Gotchas: a generator is **single-use**; `yield` defers side effects to consumption; `itertools.islice`/`batched` compose well; and batching avoids per-row overhead.

In [ ]:
# ILLUSTRATIVE - needs the live engine session and data this section names; read on for the pattern.
# Streaming pipeline: parse -> batch -> write with memory bounded by batch size
import csv
import itertools
from collections.abc import Iterable, Iterator
from typing import Any


def read_rows(path: str) -> Iterator[dict[str, str]]:
    with open(path, newline="", encoding="utf-8") as handle:
        yield from csv.DictReader(handle)


def batched(items: Iterable[Any], size: int) -> Iterator[list[Any]]:
    iterator = iter(items)
    while chunk := list(itertools.islice(iterator, size)):
        yield chunk


def run(path: str, batch_size: int = 10_000) -> int:
    return sum(write_batch(batch) for batch in batched(read_rows(path), batch_size))

Each stage is independently testable with a list, and peak memory tracks `batch_size` rather than the file.

**Mechanism.** The choice follows the bottleneck. A thread waits on I/O while the GIL is released, a process has its own interpreter and its own GIL, and async I/O multiplexes many waits on one thread with an event loop.

**Worked example.** 100 HTTP fetches at 50 ms each take about 5 s one at a time. A 100-worker thread pool overlaps the waits and finishes in roughly 50 ms. The same 100 CPU tasks would not speed up, because the GIL serialises bytecode.

**Common mistake.** Reaching for threads to fix a slow numeric loop, or starting one process per row. Match the pool to the work, cap it to the machine, and measure before you tune.

## What is the difference between threads, multiprocessing, and async IO in Python?

Threads are useful for IO-bound workloads (network, disk) but limited by the GIL for CPU-bound tasks. Multiprocessing runs multiple processes and can use multiple CPU cores for CPU-bound work. Async IO uses an event loop to handle many concurrent IO operations efficiently when tasks spend time waiting.

In [ ]:
try:
    # Async IO: thousands of concurrent requests without thousands of threads
    import asyncio

    import httpx


    async def fetch(client: httpx.AsyncClient, url: str) -> dict:
        response = await client.get(url, timeout=30)
        response.raise_for_status()
        return response.json()


    async def fetch_many(urls: list[str], concurrency: int = 100) -> list[dict]:
        limits = httpx.Limits(max_connections=concurrency)
        async with httpx.AsyncClient(limits=limits) as client:
            return await asyncio.gather(*(fetch(client, url) for url in urls))
except ModuleNotFoundError as exc:
    print(f"SKIPPED - {getattr(exc, 'name', None) or 'module'} is not installed; install it and re-run this cell.")

For CPU-bound work swap in `ProcessPoolExecutor`; for blocking IO that is not async-friendly, a `ThreadPoolExecutor` with bounded workers is often the simplest correct choice. Measure whether the workload is IO-bound or CPU-bound first.

## What is the GIL and why does it matter?

The Global Interpreter Lock (GIL) allows only one thread to execute Python bytecode at a time in CPython. This limits CPU-bound parallelism with threads. Many data engineering tasks are IO-bound, so threads can still help, but CPU-bound tasks often require multiprocessing or native libraries.

Recent nuance: CPython 3.13 introduced an **experimental free-threaded build** (PEP 703) where the GIL can be disabled, though it is not the default; the GIL is **released** around blocking IO and inside many C extensions (NumPy, Arrow, compression); and it is a CPython implementation detail, not a language rule.

In [ ]:
# If 4 threads take ~4x one thread, the GIL is serializing pure-Python CPU work
import time
from concurrent.futures import ThreadPoolExecutor


def cpu_task(n: int) -> int:
    return sum(i * i for i in range(n))


start = time.perf_counter()
with ThreadPoolExecutor(max_workers=4) as pool:
    list(pool.map(cpu_task, [2_000_000] * 4))
print(f"4 threads took {time.perf_counter() - start:.2f}s")

When the hot path sits in a GIL-releasing native library, threading scales much better; once it does not, move the work to processes or a distributed engine.

**Mechanism.** CPython keeps one lock around its interpreter state, so only one thread runs bytecode at a time. C extensions release that lock during long native calls, which is why NumPy, Arrow, and compression can still run in parallel.

**Worked example.** Four threads each summing two million squares take about as long as one thread doing four times the work. Four processes each take about a quarter, because each process owns an interpreter.

**Common mistake.** Assuming any threaded code is parallel. Check whether the hot loop is pure Python. If it is, use processes or move the loop into a native library.

## How do you read and write Parquet efficiently in Python?

Using `pyarrow` is common because it supports efficient columnar reads, predicate pushdown (engine-dependent), and schema handling. You should select only needed columns, write reasonably sized row groups/files, and avoid creating many tiny files. For large datasets, integrate with a distributed engine rather than pure local processing.

In [ ]:
try:
    # Column projection + pushed-down filter, then a typed write
    import pyarrow as pa
    import pyarrow.compute as pc
    import pyarrow.dataset as ds
    import pyarrow.parquet as pq

    dataset = ds.dataset("s3://acme-curated/orders/", format="parquet", partitioning=["order_date"])
    table = dataset.to_table(
        columns=["order_id", "customer_id", "amount"],
        filter=(pc.field("order_date") == "2026-10-04") & (pc.field("amount") > 0),
    )

    schema = pa.schema([("order_id", pa.string()), ("customer_id", pa.string()),
                        ("amount", pa.decimal128(12, 2))])
    with pq.ParquetWriter("orders/ds=2026-10-04/part-00000.parquet", schema,
                          compression="zstd", use_dictionary=True) as writer:
        for batch in table.select(schema.names).to_batches(max_chunksize=100_000):
            writer.write_batch(batch)
except ModuleNotFoundError as exc:
    print(f"SKIPPED - {getattr(exc, 'name', None) or 'module'} is not installed; install it and re-run this cell.")

Rules: never write one file per row from a local process; size files at 128-512 MB; use `zstd` cold and `snappy` hot; read with an explicit schema in production; use `pyarrow.fs`/`fsspec` for remote paths; and hand off to Spark/Polars when the work needs shuffles or hundreds of GB.

**Mechanism.** Parquet is columnar with per-column chunks and statistics, so a reader can skip row groups and read only the columns a query needs. Predicate pushdown is a property of the reader, not of the file format alone.

**Worked example.** A table with 50 columns is filtered by `order_date = '2026-10-04'`. Reading only `order_id`, `customer_id`, and `amount` for the matching row group touches a small fraction of the bytes on disk.

**Common mistake.** Writing thousands of tiny files, one per record. Every file carries a footer and metadata, so planning cost explodes. Aim for files in the 128-512 MB range.

## How do you process large CSV files without running out of memory?

You can stream the file in chunks (for example using chunked readers), process line-by-line, or use a library that supports streaming and type inference control. You also avoid loading wide columns you do not need, and you write results incrementally. For very large data, consider Parquet and/or distributed processing.

In [ ]:
try:
    # pandas chunks: aggregate incrementally, never hold the whole file
    import io

    import pandas as pd

    sample_csv = io.StringIO(
        "order_id,amount,country\n"
        "1001,49.50,DE\n1002,12.00,US\n1003,7.25,DE\n"
        "1004,99.00,US\n1005,8.80,DE\n1006,3.20,FR\n"
    )

    agg: dict[str, float] = {}
    for chunk in pd.read_csv(
        sample_csv,
        usecols=["order_id", "amount", "country"],
        dtype={"amount": "float64", "country": "category"},
        chunksize=2,   # tiny so several chunks are produced on the small fixture
    ):
        for country, total in chunk.groupby("country", observed=True)["amount"].sum().items():
            agg[country] = agg.get(country, 0.0) + total
    print("aggregate after each chunk:", agg)
except ModuleNotFoundError as exc:
    print(f"SKIPPED - {getattr(exc, 'name', None) or 'module'} is not installed; install it and re-run this cell.")

```bash
# One-off scale: SQL over a CSV larger than memory (moves parsing into DuckDB)
duckdb -c "COPY (SELECT country, sum(amount) FROM read_csv_auto('orders_2026.csv') GROUP BY country) TO 'by_country.parquet' (FORMAT PARQUET);"
```

The theme is to aggregate or filter incrementally and emit as you go, so peak memory depends on chunk size and aggregation state rather than file size. Set explicit dtypes, because inference is itself a memory and time cost.

**Mechanism.** A CSV reader is a stream: it yields one row at a time and the memory cost is the row plus the buffer, not the file. Aggregating as you read keeps only the result state, such as one total per country.

**Worked example.** Reading a 40 GB file in one-million-row chunks and adding each chunk's `groupby` total into a dict uses memory proportional to the number of countries, not to the file size.

**Common mistake.** Calling `read_csv` without `chunksize` and then filtering. The full file is parsed into memory before any filter runs. Filter and aggregate inside the chunk loop instead.

## How do you implement retries with exponential backoff?

Retries should include exponential backoff and jitter to avoid thundering herds. You should only retry on transient errors (timeouts, rate limits) and limit total retry time. Always make operations idempotent or use deduplication to prevent duplicates on retries.

In [ ]:
# ILLUSTRATIVE - needs the live engine session and data this section names; read on for the pattern.
# Capped exponential backoff with full jitter, or use tenacity's
# wait_exponential_jitter + retry_if_exception_type for the declarative version
import random
import time
from collections.abc import Callable
from typing import TypeVar

T = TypeVar("T")


class TransientError(Exception):
    """Raised for retryable failures (timeouts, 429, 503)."""


def retry(fn: Callable[[], T], *, attempts: int = 5, base: float = 0.5, cap: float = 30.0) -> T:
    last: Exception | None = None
    for attempt in range(1, attempts + 1):
        try:
            return fn()
        except TransientError as exc:
            last = exc
            if attempt == attempts:
                break
            time.sleep(random.uniform(0, min(cap, base * 2 ** (attempt - 1))))
    raise RuntimeError(f"retries exhausted after {attempts} attempts") from last

Rules: retry only transient classes; honor `Retry-After` on 429/503; cap total time, not just attempts; use idempotency keys on writes; add jitter; and log every attempt with the attempt number.

**Mechanism.** Transient failures are independent events, so retrying at once can hit the same overloaded server. Backoff spaces attempts out, and jitter spreads clients so they do not synchronise into a thundering herd.

**Worked example.** With base 0.5 s, factor 2, cap 30 s, and full jitter, the wait is `random.uniform(0, min(30, 0.5 * 2 ** attempt))`. A burst of retries spreads across the whole window instead of landing on one instant.

**Common mistake.** Retrying a `ValueError` or an HTTP 400. Retry only timeouts, 429, and 5xx, and only when the write is idempotent or carries an idempotency key.

## What logging practices are important for data pipelines?

Logs should include pipeline identifiers, run ids, input/output counts, and key timestamps. Structured logging (JSON) helps search and correlation. You should log errors with context and avoid logging sensitive data.

In [ ]:
# JSON logging with run-scoped context fields
import json
import logging
from typing import Any

logging.basicConfig(level=logging.INFO, format="%(message)s")
log = logging.getLogger("orders")


class JsonFormatter(logging.Formatter):
    def format(self, record: logging.LogRecord) -> str:
        payload: dict[str, Any] = {"level": record.levelname,
                                   "logger": record.name,
                                   "message": record.getMessage()}
        payload.update(getattr(record, "extra_fields", {}))
        if record.exc_info:
            payload["traceback"] = self.formatException(record.exc_info)
        return json.dumps(payload)


handler = logging.StreamHandler()
handler.setFormatter(JsonFormatter())
log.handlers = [handler]
log.info("stage_complete", extra={"extra_fields": {"run_id": "r-123", "rows_in": 1000, "rows_out": 998}})

Standards that matter: one start/end line per stage with `run_id`, dataset, and partition; counts in and out; `logging` with a JSON formatter rather than `print`; never log secrets or raw PII; `exc_info=True` for exceptions; and a correlation id so logs join to traces and the control table.

**Mechanism.** `print` writes unstructured text to one stream. `logging` gives named loggers, levels, and formatters, so a handler can emit one JSON object per event that a log platform can index field by field.

**Worked example.** One JSON line per stage with `run_id`, `dataset`, `partition`, `rows_in`, `rows_out`, and `duration_ms`. Finding a failed run becomes a filter on `run_id`, not a regex over prose.

**Common mistake.** Logging secrets or raw PII, and logging inside a tight row loop. Log per batch or per stage, and never log a credential.

## How do you structure a Python project for data pipelines?

A common structure separates reusable library code from job entrypoints. You keep configuration separate, use dependency injection for external services, and standardize logging and error handling. Small CLI entrypoints improve local runs and CI tests.

A common layout is `src/<pkg>/` with `config.py`, `io/`, `transforms/` (pure functions), and `jobs/` (thin CLI entrypoints), plus `tests/unit`, `tests/integration`, and `dags/` where Airflow files import the package.

Principles: business logic lives in importable modules while `jobs/` is a thin shell; config is parsed into a typed object once (`Settings.from_env()` returning a frozen dataclass); clients are injected so tests can substitute fakes; there are **no import-time side effects** (DAG parsing depends on this); and the package is installed (`uv sync`) so jobs and tests import the same code.

## How do you manage dependencies and reproducible environments?

Use pinned dependencies and a lockfile (or equivalent) and isolate environments (virtualenv/containers). Reproducibility also requires pinning Python versions and building deterministic images. CI should use the same environment definition as production.

```bash
# Reproducible workflow: lock once, install exactly what CI and prod use
uv lock                         # writes uv.lock with hashes
uv sync --frozen                # install exactly the lockfile
uv run pytest                   # tests run in the pinned environment
```

```yaml
# pyproject.toml (excerpt, shown as YAML for readability)
project:
  name: acme-data
  requires-python: ">=3.12"
  dependencies: ["pyarrow>=17,<18", "polars>=1.0,<2", "httpx>=0.27,<1"]
  optional-dependencies:
    spark: ["pyspark==3.5.1"]
    dev: ["pytest>=8", "ruff", "mypy"]
```

Anti-patterns: unpinned transitive dependencies (behavior changes silently on rebuild); installing packages at task runtime; different environments for CI and production; and hand-maintaining hundreds of `==` pins instead of using a lockfile tool. Container images should pin the base Python by digest, `COPY uv.lock`, run `uv sync --frozen`, and then copy `src` - the same lock the tests use.

## How do you test data transformations in Python?

Use small fixture datasets and assert expected outputs (golden tests). Add property tests for invariants (non-negative amounts, uniqueness). Integration tests validate real connectors and schemas, but they should be isolated and stable.

Keep transformations **pure** so the fast unit layer covers most logic, then add one integration test per connector behind a marker.

In [ ]:
try:
    # tests/unit/test_orders.py
    import pandas as pd
    import pytest

    from acme_data.transforms.orders import add_tax


    @pytest.fixture
    def orders() -> pd.DataFrame:
        return pd.DataFrame({"order_id": ["o1", "o2"], "amount_cents": [1000, 2000],
                             "country": ["DE", "US"]})


    def test_add_tax(orders):
        out = add_tax(orders, rates={"DE": 0.19, "US": 0.0})
        assert out["amount_cents"].tolist() == [1190, 2000]


    # Property test: for any list of non-negative amounts the tax never decreases them
    # @given(st.lists(st.integers(min_value=0), min_size=1))
    # def test_tax_never_decreases(amounts): ...
except ModuleNotFoundError as exc:
    print(f"SKIPPED - {getattr(exc, 'name', None) or 'module'} is not installed; install it and re-run this cell.")

```bash
uv run pytest tests/unit -q                       # always, in CI
uv run pytest -m integration --maxfail=1          # opt-in, needs a test warehouse
```

Keep fixtures small enough to verify by eye, and assert business rules rather than snapshotting current output - change-detector tests pass forever and catch nothing.

**Mechanism.** A transformation test is pure: fixed input, call the function, compare the output to a literal. Keeping I/O outside the function means the test needs no cluster, no network, and no fixtures beyond plain data.

**Worked example.** A `dedupe(rows)` function keeps the last row per key. `assert dedupe([{"id": 1, "v": 1}, {"id": 1, "v": 2}]) == [{"id": 1, "v": 2}]` runs in microseconds and names the exact behaviour that broke.

**Common mistake.** Testing through a full pipeline run and asserting on a dashboard. It is slow, flaky, and does not localise the bug. Test the small function, then test the wiring separately.

## How do you profile and optimize slow Python code?

Profile first to find hotspots (CPU, IO, memory). Common fixes include vectorizing operations, reducing data copies, using efficient data structures, and offloading heavy compute to native libraries or distributed engines. Optimization should be guided by measurable impact.

Pick the tool by question: `cProfile` + `pstats` for wall-clock hotspots, `line_profiler` for a specific line, `tracemalloc`/`memray` for peak or leaking memory, and `py-spy dump` to see whether a running process is IO- or CPU-bound.

In [ ]:
# cProfile + pstats: sort by cumulative time to find the real hotspot
import cProfile
import pstats


def run_pipeline(ds: str) -> int:
    """Stand-in for the real pipeline so the profiler demo is runnable."""
    return sum(i * i for i in range(20_000))


def main() -> None:
    run_pipeline("2026-10-04")


profiler = cProfile.Profile()
profiler.enable()
main()
profiler.disable()
pstats.Stats(profiler).sort_stats("cumulative").print_stats(20)

Common wins in order of impact: vectorize row-wise `.apply`/loops (`(df["amount"] * (1 + df["tax_rate"])).sum()` instead of `iterrows`); avoid copies (`copy=False`, reuse buffers, prefer Arrow over `to_list()`); choose the right container; reduce IO by batching and streaming; and move compute to SQL/Spark when the data never needed to be in Python.

## How do you write memory-safe Python ETL code?

Memory-safe ETL code avoids loading unbounded data into lists or DataFrames. Prefer iterators, chunked reads, streaming writes, and explicit column selection. This is especially important for scripts that run in containers with fixed memory limits.

Example:

In [ ]:
import csv
import tempfile

def active_users(path):
    with open(path, newline="") as f:
        reader = csv.DictReader(f)
        for row in reader:
            if row["status"] == "active":
                yield {
                    "user_id": int(row["user_id"]),
                    "email": row["email"].lower(),
                }

def send_to_api(user):
    print("sent to API:", user["email"])   # stand-in for the real upload

# write the fixture a production run would stream from a drop folder or object store
with tempfile.NamedTemporaryFile("w", suffix=".csv", newline="", delete=False) as fh:
    writer = csv.DictWriter(fh, fieldnames=["user_id", "email", "status"])
    writer.writeheader()
    writer.writerows(
        [
            {"user_id": 1, "email": "ana@example.io", "status": "active"},
            {"user_id": 2, "email": "bob@example.io", "status": "inactive"},
            {"user_id": 3, "email": "cy@example.io", "status": "active"},
        ]
    )
    users_path = fh.name

for user in active_users(users_path):
    send_to_api(user)

This processes one row at a time instead of building a full in-memory collection. In interviews, call out the trade-off: row-by-row code is memory efficient, while vectorized/batch processing can be faster when data fits in memory.

Checklist: bound every buffer (chunk, batch, and queue sizes); select columns in the reader rather than dropping later; use `category` dtype for low-cardinality strings and downcast numerics; never `concat` inside a loop (collect and concat once, or stream to disk); and cap concurrency so parallel fetches cannot exceed the container limit. Test with a larger-than-expected input, because a pipeline that only works on the sample partition is an outage waiting to happen.

## How do you design Python code so a data pipeline is restartable?

A restartable pipeline can safely resume after failure without corrupting downstream data. Use idempotent writes, deterministic output paths, checkpoints/control tables, and stable run identifiers. Avoid hidden global state and make external side effects explicit.

Example pattern:

In [ ]:
# ILLUSTRATIVE - needs the live engine session and data this section names; read on for the pattern.
def run_partition(ds: str) -> None:
    source = f"s3://raw/events/ds={ds}/"
    staging = f"s3://warehouse/events_staging/run_ds={ds}/"
    final = f"s3://warehouse/events/ds={ds}/"

    records = transform(read_json(source))
    write_parquet(records, staging)
    validate_partition(staging)
    replace_partition(staging, final)
    mark_success("events", ds)

The important idea is to write to staging first, validate, then atomically publish or replace the final partition. If the job retries, it produces the same partition instead of appending duplicates. A control table (`platform.partition_state` with dataset, partition, status, attempt, rows_written, finished_at) tracks per-partition state so the loop can skip completed work.

Rules that make restart safe: deterministic paths (same logical date, same output); atomic publish via rename, partition overwrite, or MERGE; a stable run id passed through every stage; explicit enumeration of external side effects; and no mutable module-level state that assumes one run per process.

## How do you handle bad records in Python pipelines?

Do not let one malformed row either crash an entire batch blindly or disappear silently. A strong pipeline separates valid records from rejected records, records the reason for rejection, and emits metrics so data quality issues are visible.

In [ ]:
# ILLUSTRATIVE - needs the live engine session and data this section names; read on for the pattern.
# Dead-letter records carry enough context to replay, plus a metric
from dataclasses import asdict, dataclass
from datetime import datetime, timezone
from typing import Any


@dataclass
class DeadLetter:
    source_file: str
    line_number: int
    error_message: str
    payload: str
    detected_at: str


def parse_with_dlq(row: dict[str, str], source_file: str, line_number: int) -> dict[str, Any] | None:
    try:
        return {"order_id": row["order_id"], "amount": float(row["amount"])}
    except (KeyError, ValueError) as exc:
        dead_letter.write(asdict(DeadLetter(
            source_file=source_file, line_number=line_number, error_message=str(exc),
            payload=json.dumps(row)[:4000], detected_at=datetime.now(timezone.utc).isoformat(),
        )))
        REJECTED.labels(pipeline="orders").inc()
        return None

In production, rejected records go to a dead-letter path/table with the source file, line number, pipeline version, and error message, so teams can fix upstream data or replay rejects after correction. Enforce a reject-rate threshold so a flood of malformed records blocks rather than silently publishing. **Failure mode:** swallowing bad rows with `try/except: pass` - the job goes green, the metric declines, and the records cannot be replayed.

**Mechanism.** A pipeline that raises on the first bad row stops a whole partition. A quarantine path records the bad record with its error and offset, and the good rows keep flowing.

**Worked example.** A per-row `try`/`except` appends `(raw_line, error)` to a dead-letter file and increments a counter. The job finishes, reports `bad_records=3`, and the file can be replayed later.

**Common mistake.** Silently dropping bad rows, or letting one malformed row fail the run. Count and store rejects, and fail only when the reject rate crosses a threshold you chose.

## When should you use pandas, Polars, PyArrow, or Spark from Python?

Use pandas for small to medium in-memory analysis and simple transformations. Use Polars or PyArrow when you need faster columnar processing, lazy execution, or efficient Parquet/Arrow interoperability on a single machine. Use Spark/PySpark when data volume, shuffle requirements, or production architecture require distributed processing.

Example decision rule:

+ pandas: one CSV or Parquet extract that fits comfortably in memory
+ PyArrow: efficient file/schema operations, Parquet datasets, Arrow interchange
+ Polars: fast local transformations and lazy query optimization
+ PySpark: joins, aggregations, and pipelines over data too large for one machine

The interview answer should emphasize workload shape, not tool popularity. A 5 GB file may be fine in Polars on a large instance, while a 500 GB join with skewed keys usually belongs in Spark or a warehouse.

In [ ]:
try:
    # Polars: lazy scan with projection and predicate pushdown on a large local dataset
    import polars as pl

    result = (
        pl.scan_parquet("s3://acme-curated/orders/*/*.parquet")
        .filter(pl.col("order_date") == "2026-10-04")
        .group_by("country")
        .agg(pl.col("amount").sum().alias("revenue"))
        .collect(streaming=True)
    )
except ModuleNotFoundError as exc:
    print(f"SKIPPED - {getattr(exc, 'name', None) or 'module'} is not installed; install it and re-run this cell.")

Rule of thumb: **stay single-node until the data genuinely exceeds memory or the shuffle does not fit on one machine.** Polars/DuckDB push that boundary higher than pandas, and moving to Spark prematurely costs cluster operations and JVM serialization you did not need to pay.

## Real world use cases for Python for Data Engineering

### 1. E-commerce: API ingestion with retries and a dead-letter queue

A retailer pulls orders from a partner REST API that rate-limits aggressively. Roughly 4M orders/day arrive in 5-minute windows, then land in S3 and load into the warehouse.

In [ ]:
try:
    # Bounded-concurrency fetcher with retry and backoff; exhausted pages go to a DLQ
    import asyncio

    import httpx
    from tenacity import retry, retry_if_exception_type, stop_after_attempt, wait_exponential_jitter


    class RateLimited(Exception):
        pass


    @retry(retry=retry_if_exception_type((RateLimited, httpx.TransportError)),
           wait=wait_exponential_jitter(initial=1, max=60), stop=stop_after_attempt(5), reraise=True)
    async def fetch_page(client: httpx.AsyncClient, cursor: str | None) -> dict:
        response = await client.get("/orders", params={"cursor": cursor, "limit": 500})
        if response.status_code == 429:
            raise RateLimited(response.headers.get("Retry-After", "unknown"))
        response.raise_for_status()
        return response.json()
except ModuleNotFoundError as exc:
    print(f"SKIPPED - {getattr(exc, 'name', None) or 'module'} is not installed; install it and re-run this cell.")

A page that exhausts retries is written to a quarantine prefix with its cursor, so the pull resumes from the last good cursor instead of re-reading the whole day.

**Why this design:** the API is the bottleneck, so async IO plus bounded concurrency maximizes throughput, retries with jitter absorb 429s, and a durable cursor makes a failure cost one page rather than one day.

### 2. Fintech: reconciliation of two systems with Polars

A payments team reconciles ~120M transactions/day between a core ledger and the warehouse, classifying differences before the close.

In [ ]:
try:
    # Vectorized reconciliation: a set join, not a loop over 120M rows
    import polars as pl

    ledger = pl.scan_parquet("s3://acme-ledger/txn/dt=2026-10-04/*.parquet").select("txn_id", "amount_cents")
    warehouse = pl.scan_parquet("s3://acme-wh/txn/dt=2026-10-04/*.parquet").select("txn_id", "amount_cents")

    recon = (
        ledger.join(warehouse, on="txn_id", how="full", suffix="_wh")
        .with_columns(
            pl.when(pl.col("amount_cents").is_null()).then(pl.lit("missing_in_warehouse"))
            .when(pl.col("amount_cents_wh").is_null()).then(pl.lit("missing_in_ledger"))
            .when(pl.col("amount_cents") != pl.col("amount_cents_wh")).then(pl.lit("amount_mismatch"))
            .otherwise(pl.lit("match")).alias("recon_status")
        )
        .group_by("recon_status").agg(pl.len().alias("count"))
        .collect(streaming=True)
    )
except ModuleNotFoundError as exc:
    print(f"SKIPPED - {getattr(exc, 'name', None) or 'module'} is not installed; install it and re-run this cell.")

**Why this design:** reconciliation is a set operation, so a vectorized join handles 120M rows on one node in minutes, and the output classifies every difference instead of returning a binary gate the finance team cannot act on.

### 3. Multi-tenant SaaS: memory-bounded export per tenant

A B2B platform lets customers download their data. Some tenants have billions of events; the export service runs in a container with a 2 GB limit and must never OOM.

In [ ]:
# ILLUSTRATIVE - needs the live engine session and data this section names; read on for the pattern.
# Server-side cursor plus fetchmany keeps memory proportional to the batch
BATCH = 100_000


def stream_tenant_events(conn, tenant_id: str, out_path: str) -> int:
    written = 0
    with gzip.open(out_path, "wt", encoding="utf-8") as handle, conn.cursor(name="export_cursor") as cur:
        cur.itersize = BATCH
        cur.execute("SELECT event_id, event_ts FROM events WHERE tenant_id = %s", (tenant_id,))
        while rows := cur.fetchmany(BATCH):
            for row in rows:
                handle.write(json.dumps(row) + "\n")
            written += len(rows)
    return written

**Why this design:** the container limit is a hard constraint, so memory is bounded by the batch rather than the tenant, and compressed NDJSON streams bytes to disk instead of buffering them.

### 4. ML feature platform: a reproducible training snapshot

An ML team builds daily features for 40M users. Reproducibility matters because model comparisons must attribute changes to the model, not to a re-run of feature code.

In [ ]:
try:
    # Installable, versioned feature job with a content hash for the snapshot
    import hashlib

    import polars as pl


    def build_features(input_glob: str, code_version: str) -> tuple[pl.DataFrame, str]:
        frame = (
            pl.scan_parquet(input_glob)
            .group_by("user_id")
            .agg(pl.col("event_type").eq("view").sum().alias("views_1d"),
                 pl.col("amount").sum().alias("spend_1d"))
            .collect(streaming=True)
        )
        digest = hashlib.sha256(f"{code_version}{frame.shape}{frame.sum().to_dict()}".encode())
        return frame, digest.hexdigest()[:16]
except ModuleNotFoundError as exc:
    print(f"SKIPPED - {getattr(exc, 'name', None) or 'module'} is not installed; install it and re-run this cell.")

**Why this design:** the job is a normal installable package with a lockfile, so CI and production run the same code and dependency versions, and a content hash identifies the exact snapshot an experiment trained on.

## Practice: Python basics

The hands-on practice for this topic lives in its own notebook: [`01_python_basics.ipynb`](../01_python_basics.ipynb) — [Open in Colab](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/01_python_basics.ipynb).

## Practice: Python coding problems

The hands-on practice for this topic lives in its own notebook: [`02_python_coding_problems.ipynb`](../02_python_coding_problems.ipynb) — [Open in Colab](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/02_python_coding_problems.ipynb).

## Python exercises

Eight short exercises cover the mechanics that the problem set does not: defaults, closures, identity, copying, containers, errors, dataclasses, and string building. Write your answer, then run the cell to check it.

### Try it 1. Mutable default arguments

**Task.** Write `add_tag(tag, tags=None)` that returns a list with `tag` appended. Show that a mutable default of `[]` is shared across calls, and that `None` gives each call a fresh list.

**Hint.** Default arguments are evaluated once, when the `def` statement runs, so the same list object is reused on every call. Use `None` as the sentinel and build the list inside the body.

**Expected.** The buggy version returns `['a']` then `['a', 'b']`; the safe version returns `['a']` then `['b']`.

In [ ]:
def buggy(tag, tags=[]):
    tags.append(tag)
    return tags


def safe(tag, tags=None):
    if tags is None:
        tags = []
    tags.append(tag)
    return tags


assert buggy("a") == ["a"]
assert buggy("b") == ["a", "b"]   # the default list is shared
assert safe("a") == ["a"]
assert safe("b") == ["b"]         # a fresh list on each call
print("buggy:", buggy("c"), "| safe:", safe("c"))

### Try it 2. Closure late binding

**Task.** Build a list of lambdas in a loop that each return their own loop value, not the final value. Compare a late-binding version with a default-argument fix and with `functools.partial`.

**Hint.** A closure captures the variable, not its value at definition time, so all the lambdas see the loop variable after the loop ends. Bind the value with a default argument or `partial`.

**Expected.** The naive lambdas all return `2`; the fixed ones return `0, 1, 2`.

In [ ]:
from functools import partial

late = [lambda: i for i in range(3)]
assert [f() for f in late] == [2, 2, 2]

bound = [lambda i=i: i for i in range(3)]
assert [f() for f in bound] == [0, 1, 2]

partials = [partial(lambda i: i, i) for i in range(3)]
assert [f() for f in partials] == [0, 1, 2]
print("late:", [f() for f in late], "| bound:", [f() for f in bound])

### Try it 3. `is` versus `==`

**Task.** Show the difference between identity and equality for a list alias, two equal dicts, and `None`. Use `is` only for singletons.

**Hint.** `==` asks whether two objects compare equal; `is` asks whether they are the same object in memory. CPython also caches small integers, so `256 is 256` is true by implementation detail.

**Expected.** The alias is identical, the equal dicts are not, and `None` is only safely tested with `is`.

In [ ]:
row = {"id": 1}
alias = row
assert alias is row                 # same object
assert {"id": 1} == row             # equal value
assert {"id": 1} is not row         # a new object

value = None
assert value is None                # identity is the right test for None
assert value == None                # works, but style tools flag it

small = 256
cached = 256
assert small is cached              # CPython caches -5..256
print("alias is row:", alias is row, "| equal dicts:", {"id": 1} == row)

### Try it 4. Shallow versus deep copy

**Task.** Copy a nested list with `copy.copy` and `copy.deepcopy`. Mutate an inner list through the shallow copy and show that the original changes; show that the deep copy stays independent.

**Hint.** `copy.copy` builds a new outer container but reuses the inner objects. `copy.deepcopy` walks the whole object graph and copies every level.

**Expected.** After `shallow[0].append(99)`, the original first row contains `99`; the deep copy does not.

In [ ]:
import copy

matrix = [[1, 2], [3, 4]]
shallow = copy.copy(matrix)
deep = copy.deepcopy(matrix)

shallow[0].append(99)
assert matrix[0] == [1, 2, 99]      # the inner list is shared
assert deep[0] == [1, 2]            # the deep copy is independent

deep[1].append(7)
assert matrix[1] == [3, 4]
print("matrix:", matrix, "| deep:", deep)

### Try it 5. Dict ordering and `collections`

**Task.** Count words into a plain `dict` and confirm insertion order, then use `Counter`, `defaultdict`, and `OrderedDict.move_to_end`.

**Hint.** Since Python 3.7 `dict` keeps insertion order as a language guarantee. `Counter` counts, `defaultdict` creates missing values, and `OrderedDict` has order-aware methods.

**Expected.** Keys come back in first-seen order; `Counter.most_common(2)` returns the two most frequent words; `move_to_end` reorders the `OrderedDict`.

In [ ]:
from collections import Counter, OrderedDict, defaultdict

words = "b a b c a b".split()
counts = {}
for word in words:
    counts[word] = counts.get(word, 0) + 1
assert list(counts) == ["b", "a", "c"]        # insertion order is preserved
assert dict(sorted(counts.items())) == {"a": 2, "b": 3, "c": 1}

counter = Counter(words)
assert counter.most_common(2) == [("b", 3), ("a", 2)]

groups = defaultdict(list)
for key, value in [("us", 1), ("eu", 2), ("us", 3)]:
    groups[key].append(value)
assert dict(groups) == {"us": [1, 3], "eu": [2]}

ordered = OrderedDict([("first", 1), ("second", 2)])
ordered.move_to_end("first")
assert list(ordered) == ["second", "first"]
print("order:", list(counts), "| top:", counter.most_common(2))

### Try it 6. Exception chaining

**Task.** Catch a low-level `FileNotFoundError` and raise a `ConfigError` with `raise ... from`. Inspect `__cause__`. Then raise `from None` and confirm the chain is suppressed.

**Hint.** `raise New from original` sets `__cause__` and keeps the traceback honest. `raise ... from None` sets `__suppress_context__` and hides the original context.

**Expected.** The first error reports a `FileNotFoundError` cause; the second has `__cause__ is None`.

In [ ]:
class ConfigError(Exception):
    pass


def load(path):
    try:
        raise FileNotFoundError(path)
    except FileNotFoundError as exc:
        raise ConfigError(f"cannot load {path}") from exc


try:
    load("settings.yaml")
except ConfigError as exc:
    assert isinstance(exc.__cause__, FileNotFoundError)
    assert "settings.yaml" in str(exc)
    print("chained:", type(exc).__name__, "<-", type(exc.__cause__).__name__)

try:
    try:
        raise ValueError("inner")
    except ValueError:
        raise RuntimeError("clean") from None
except RuntimeError as exc:
    assert exc.__cause__ is None and exc.__suppress_context__ is True
print("exception chaining checks passed")

### Try it 7. Dataclass versus a hand-written `__init__`

**Task.** Compare a class with a hand-written `__init__` and `__eq__` against an equivalent `@dataclass`. Cover equality, `repr`, `default_factory`, and `frozen=True`.

**Hint.** `@dataclass` generates `__init__`, `__repr__`, and `__eq__` from the annotations. `field(default_factory=list)` gives each instance its own list, and `frozen=True` blocks attribute assignment.

**Expected.** The dataclass prints its fields, equal instances compare equal, each instance owns a separate list, and assignment to a frozen instance raises `FrozenInstanceError`.

In [ ]:
from dataclasses import FrozenInstanceError, dataclass, field


class Manual:
    def __init__(self, order_id, amount):
        self.order_id = order_id
        self.amount = amount

    def __eq__(self, other):
        return (self.order_id, self.amount) == (other.order_id, other.amount)


@dataclass
class Order:
    order_id: str
    amount: float
    tags: list = field(default_factory=list)


assert Manual("o1", 10.0) == Manual("o1", 10.0)
assert "Manual object" in repr(Manual("o1", 10.0))    # no auto repr

order_a = Order("o1", 10.0)
order_b = Order("o1", 10.0)
assert order_a == order_b
assert repr(order_a) == "Order(order_id='o1', amount=10.0, tags=[])"
order_a.tags.append("priority")
assert order_b.tags == []                             # one list per instance


@dataclass(frozen=True)
class Point:
    x: int
    y: int


try:
    Point(1, 2).x = 5
except FrozenInstanceError as exc:
    print("frozen:", type(exc).__name__)
print("dataclass repr:", order_a)

### Try it 8. String joining versus concatenation

**Task.** Build one long string with a `+=` loop and with `"".join(...)`. Assert that both give the same text, and print the timing of each as a guide.

**Hint.** Strings are immutable, so `+=` builds a new string and copies everything so far, which is quadratic in the number of parts. `join` walks the parts once and allocates the result once.

**Expected.** The two results are equal, and the printed timings show `join` doing the same work without repeated copies.

In [ ]:
import time


def concat(parts):
    result = ""
    for part in parts:
        result += part
    return result


def join(parts):
    return "".join(parts)


words = [f"row-{i}," for i in range(20_000)]
assert concat(words[:100]) == join(words[:100])

started = time.perf_counter()
joined = join(words)
join_ms = (time.perf_counter() - started) * 1000

started = time.perf_counter()
built = concat(words)
concat_ms = (time.perf_counter() - started) * 1000

assert joined == built
print(f"join {join_ms:.2f} ms | concat {concat_ms:.2f} ms")
print("join builds one buffer; += copies the accumulated string each time")